# BattINFO schema versions

The BattINFO records on `Metadata` (`battinfo_test`, `battinfo_cell`, ...) are bounded by a loaded BattINFO JSON Schema set, `bdf.battinfo.BATTINFO_SCHEMA`. Like `COLUMN_ONTOLOGY`, it is built at import time from the snapshot of one upstream release bundled with the package, and it can adopt another set in place.

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

import bdf
from bdf.battinfo import BATTINFO_SCHEMA, BattinfoSchema, SchemaError
from bdf.battinfo.schema import bundled_schema_dir

print("loaded BattINFO release:", BATTINFO_SCHEMA.version)
print("documents:", sorted(BATTINFO_SCHEMA.docs))

## A record is bounded by the loaded set

A record offers the fields the loaded set declares, so `dir()` and tab completion list them. A value the set refuses raises `SchemaError`, which names the release.

In [ ]:
test = bdf.Metadata().battinfo_test.test
print([name for name in dir(test) if not name.startswith("_")])

try:
    test.kind = "ocv"
except SchemaError as exc:
    print(str(exc)[:120], "...")

## Load an in-development schema set

`load_dir(path)` adopts the schema documents under any local directory, such as a BattINFO checkout with a change that is not released yet. Here we copy the bundled documents and add `ocv` to the test kinds. The adopted set names no release.

Records built after the change are bounded by the adopted set.

In [ ]:
dev = Path(tempfile.mkdtemp()) / "schemas"
shutil.copytree(bundled_schema_dir(), dev)
test_doc = dev / "test.schema.json"
doc = json.loads(test_doc.read_text(encoding="utf-8"))
doc["properties"]["test"]["properties"]["kind"]["enum"].append("ocv")
test_doc.write_text(json.dumps(doc, indent=2), encoding="utf-8")

BATTINFO_SCHEMA.load_dir(dev)
print("loaded BattINFO release:", BATTINFO_SCHEMA.version)

meta = bdf.Metadata()
meta.battinfo_test.test.kind = "ocv"
print("test kind:", meta.battinfo_test.test.kind)

## Load a published release

`load_version(version)` loads one upstream BattINFO release, by its package version. It serves the cached copy when present, and otherwise fetches the release's tag and caches it under `BDF_CACHE_DIR` for next time. `refresh=True` fetches it again.

Loading the bundled release here also undoes the change above: the release refuses `ocv` again.

In [ ]:
BATTINFO_SCHEMA.load_version("0.8.0rc2")
print("loaded BattINFO release:", BATTINFO_SCHEMA.version)

try:
    bdf.Metadata().battinfo_test.test.kind = "ocv"
except SchemaError as exc:
    print(str(exc)[:120], "...")

`load_latest(refresh=True)` asks upstream for its latest release, the newest release tag, and loads it. Use it to pick up a release newer than the bundled snapshot. Here we use a fresh set from `BattinfoSchema.build()`, so the shared `BATTINFO_SCHEMA` is unchanged. (This cell requires network access.)

In [ ]:
latest = BattinfoSchema.build()
try:
    latest.load_latest(refresh=True)
    print("latest BattINFO release:", latest.version)
except Exception as exc:
    print("latest release unavailable, keeping the bundled snapshot:", type(exc).__name__)

The loaded set governs the records you build. A file read back with `bdf.read` is different: its sidecar names the BattINFO release it was written with (`bdf.battinfo_version`), and its records are checked against that release, whichever set is loaded.